In [ ]:
pip install torch

In [ ]:
import torch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

In [ ]:
b

tensor([[1., 2.],
        [3., 4.]])

In [ ]:
x1 = torch.randn(3,4)
x2 = torch.ones(4)

x1 @ x2

tensor([-0.6252,  1.3932,  0.0485])

In [ ]:
W = torch.randn(3, 4, requires_grad=True)
x = torch.ones(4)
x_new = x @ W.T
loss = x_new.sum()
loss.backward()
W.grad

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

train_dataset = torchvision.datasets.MNIST(
    root = './sample_data',
    train = True,
    transform = transforms.ToTensor(),
    download = True
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size = 64, shuffle = True
)

W1 = (torch.randn(256, 784) * 0.01).to(device).requires_grad_(True)
b1 = torch.zeros(256).to(device).requires_grad_(True)

W2 = (torch.randn(128, 256) * 0.01).to(device).requires_grad_(True)
b2 = torch.zeros(128).to(device).requires_grad_(True)

W3 = (torch.randn(10, 128) * 0.01).to(device).requires_grad_(True)
b3 = torch.zeros(10).to(device).requires_grad_(True)

learning_rate = 0.01

for epoch in range(7):
  total_loss = 0

  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    x = images.reshape(-1, 784)

    h1 = F.relu(x @ W1.T + b1)
    h2 = F.relu(h1 @ W2.T + b2)
    out = h2 @ W3.T + b3

    loss = F.cross_entropy(out, labels)

    loss.backward()

    with torch.no_grad():
      W1 -= learning_rate * W1.grad
      b1 -= learning_rate * b1.grad
      W2 -= learning_rate * W2.grad
      b2 -= learning_rate * b2.grad
      W3 -= learning_rate * W3.grad
      b3 -= learning_rate * b3.grad

    W1.grad.zero_()
    b1.grad.zero_()
    W2.grad.zero_()
    b2.grad.zero_()
    W3.grad.zero_()
    b3.grad.zero_()

    total_loss += loss.item()
  print(f"Epoch {epoch+1}, loss = {total_loss/len(train_loader):.4f}")




Epoch 1, loss = 2.3006
Epoch 2, loss = 2.2893
Epoch 3, loss = 1.9090
Epoch 4, loss = 0.9184
Epoch 5, loss = 0.6339
Epoch 6, loss = 0.5090
Epoch 7, loss = 0.4353


In [ ]:
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False,
    transform=transforms.ToTensor(), download=True
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=64, shuffle=False
)

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        x = images.reshape(-1, 784)

        h1 = torch.relu(x @ W1.T + b1)
        h2 = torch.relu(h1 @ W2.T + b2)
        out = h2 @ W3.T + b3

        preds = out.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

accuracy = (all_preds == all_labels).float().mean()
print(f"Accuracy: {accuracy:.4f}")

from sklearn.metrics import f1_score, classification_report
f1 = f1_score(all_labels.numpy(), all_preds.numpy(), average='macro')
print(f"F1 macro: {f1:.4f}")

print(classification_report(all_labels.numpy(), all_preds.numpy()))

Accuracy: 0.8837
F1 macro: 0.8812
              precision    recall  f1-score   support

           0       0.92      0.97      0.95       980
           1       0.95      0.97      0.96      1135
           2       0.91      0.87      0.89      1032
           3       0.85      0.89      0.87      1010
           4       0.86      0.90      0.88       982
           5       0.83      0.78      0.80       892
           6       0.91      0.92      0.91       958
           7       0.91      0.89      0.90      1028
           8       0.86      0.78      0.82       974
           9       0.82      0.84      0.83      1009

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Данные
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True,
    transform=transforms.ToTensor(), download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False,
    transform=transforms.ToTensor(), download=True
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = x.reshape(-1, 784)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

model = Net().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

def train(epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()      # обнуляем градиенты
            out = model(images)        # forward pass
            loss = loss_fn(out, labels)
            loss.backward()            # backward
            optimizer.step()           # обновляем веса

            total_loss += loss.item()

        model.eval()  # режим оценки — dropout выключается
        correct = 0
        total = 0

        with torch.no_grad():
          for images, labels in test_loader:
              images, labels = images.to(device), labels.to(device)
              out = model(images)
              preds = out.argmax(dim=1)
              correct += (preds == labels).sum().item()
              total += labels.size(0)

        print(f"Epoch {epoch+1} | loss: {total_loss/len(train_loader):.4f} | test accuracy: {correct/total:.4f}")

train(10)

Epoch 1 | loss: 0.3002 | test accuracy: 0.9632
Epoch 2 | loss: 0.1285 | test accuracy: 0.9742
Epoch 3 | loss: 0.0990 | test accuracy: 0.9746
Epoch 4 | loss: 0.0833 | test accuracy: 0.9800
Epoch 5 | loss: 0.0718 | test accuracy: 0.9781
Epoch 6 | loss: 0.0622 | test accuracy: 0.9750
Epoch 7 | loss: 0.0578 | test accuracy: 0.9819
Epoch 8 | loss: 0.0505 | test accuracy: 0.9826
Epoch 9 | loss: 0.0459 | test accuracy: 0.9797
Epoch 10 | loss: 0.0471 | test accuracy: 0.9792


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):
        d_k = Q.shape[-1]

        # scores shape: (batch, heads, seq_len, seq_len)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

        # маска для decoder — заполняем -inf чтобы после softmax стало 0
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)

        # output shape: (batch, heads, seq_len, d_k)
        return weights @ V, weights

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, h=8):
        super().__init__()
        assert d_model % h == 0  # d_model должен делиться на h

        self.h = h
        self.d_k = d_model // h  # 64

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

    def forward(self, Q, K, V, mask=None):
        batch = Q.shape[0]

        # Проецируем через W_Q, W_K, W_V
        Q = self.W_Q(Q)  # (batch, seq_len, d_model)
        K = self.W_K(K)
        V = self.W_V(V)

        # Разбиваем на головы
        Q = Q.reshape(batch, -1, self.h, self.d_k).transpose(1, 2)
        K = K.reshape(batch, -1, self.h, self.d_k).transpose(1, 2)
        V = V.reshape(batch, -1, self.h, self.d_k).transpose(1, 2)
        # теперь shape: (batch, h, seq_len, d_k)

        # Считаем attention для всех голов параллельно
        out, weights = self.attention(Q, K, V, mask)
        # out shape: (batch, h, seq_len, d_k)

        # Собираем головы обратно
        out = out.transpose(1, 2)  # (batch, seq_len, h, d_k)
        out = out.reshape(batch, -1, self.h * self.d_k)  # (batch, seq_len, d_model)

        # Финальная проекция
        return self.W_O(out)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, x):
        x = F.relu(self.linear1(x))  # (batch, seq_len, 2048)
        x = self.dropout(x)
        x = self.linear2(x)          # (batch, seq_len, 512)
        return x

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, h)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual + norm
        attn_out = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # FeedForward + residual + norm
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, N=6, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, h, d_ff, dropout) for _ in range(N)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [ ]:
encoder = Encoder()
total = sum(p.numel() for p in encoder.parameters())
print(f"Параметров: {total:,}")

Параметров: 18,915,328


In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.masked_attention = MultiHeadAttention(d_model, h)  # self-attention с маской
        self.cross_attention = MultiHeadAttention(d_model, h)   # cross-attention
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_out, src_mask=None, tgt_mask=None):
        # Подслой 1 — masked self-attention по словам перевода
        attn_out = self.masked_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Подслой 2 — cross-attention
        # Q из decoder, K и V из encoder
        attn_out = self.cross_attention(x, encoder_out, encoder_out, src_mask)
        x = self.norm2(x + self.dropout(attn_out))

        # Подслой 3 — feedforward
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))

        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048, N=6, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, h, d_ff, dropout) for _ in range(N)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, encoder_out, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, encoder_out, src_mask, tgt_mask)
        return self.norm(x)

In [ ]:
def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)

    position = torch.arange(max_seq_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )

    PE[:, 0::2] = torch.sin(position * div_term)
    PE[:, 1::2] = torch.cos(position * div_term)

    return PE

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, h=8,
                 d_ff=2048, N=6, dropout=0.1, max_seq_len=100):
        super().__init__()

        # Эмбеддинги
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding
        self.register_buffer('pe', positional_encoding(max_seq_len, d_model))

        self.dropout = nn.Dropout(dropout)

        # Encoder и Decoder
        self.encoder = Encoder(d_model, h, d_ff, N, dropout)
        self.decoder = Decoder(d_model, h, d_ff, N, dropout)

        # Финальный слой — проецируем в словарь
        self.linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Encoder
        src = self.src_embedding(src) * math.sqrt(512)
        src = self.dropout(src + self.pe[:src.shape[1]])
        encoder_out = self.encoder(src, src_mask)

        # Decoder
        tgt = self.tgt_embedding(tgt) * math.sqrt(512)
        tgt = self.dropout(tgt + self.pe[:tgt.shape[1]])
        decoder_out = self.decoder(tgt, encoder_out, src_mask, tgt_mask)

        # Проецируем в словарь
        return self.linear(decoder_out)  # (batch, seq_len, tgt_vocab_size)

In [ ]:
model = Transformer(src_vocab_size=10000, tgt_vocab_size=10000)
src = torch.randint(0, 10000, (2, 10))  # батч 2, исходное предложение 10 токенов
tgt = torch.randint(0, 10000, (2, 8))   # батч 2, перевод 8 токенов
out = model(src, tgt)
print(out.shape)  # (2, 8, 10000) — для каждого токена вероятности по всему словарю

torch.Size([2, 8, 10000])


In [ ]:
model = Transformer(src_vocab_size=10000, tgt_vocab_size=10000).to(device)

# Считаем параметры
total = sum(p.numel() for p in model.parameters())
print(f"Параметров: {total:,}")

# Тестовый прогон
src = torch.randint(0, 10000, (2, 10)).to(device)
tgt = torch.randint(0, 10000, (2, 8)).to(device)

out = model(src, tgt)
print(f"Выход: {out.shape}")  # (2, 8, 10000)

Параметров: 59,510,544
Выход: torch.Size([2, 8, 10000])
